# Summarization — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(x, axis=-1):
    x=np.asarray(x, dtype=float); x=x-x.max(axis=axis, keepdims=True); e=np.exp(x); return e/e.sum(axis=axis, keepdims=True)
def sigmoid(x): return 1/(1+np.exp(-np.asarray(x, dtype=float)))
def normalize(v):
    v=np.asarray(v, dtype=float); n=np.linalg.norm(v); return v/(n if n else 1)
def edit_distance(a,b):
    D=np.zeros((len(a)+1,len(b)+1), dtype=int); D[:,0]=np.arange(len(a)+1); D[0,:]=np.arange(len(b)+1)
    for i in range(1,len(a)+1):
        for j in range(1,len(b)+1):
            D[i,j]=min(D[i-1,j]+1, D[i,j-1]+1, D[i-1,j-1]+(a[i-1]!=b[j-1]))
    return D
print('setup ok')

## Compress text while preserving salient information
The cells show sentence salience, redundancy penalty, compression ratio, ROUGE recall and an abstractive fusion toy.

In [ ]:
sal=np.array([0.9,0.4,0.8,0.2]); plt.figure(figsize=(5,3)); plt.bar(range(4),sal); plt.title('Extractive summarization ranks sentence salience')
assert int(np.argmax(sal))==0

In [ ]:
sim=np.array([[1,.8,.1],[.8,1,.2],[.1,.2,1]]); chosen=[0]; mmr=0.7*np.array([.9,.8,.7])-0.3*sim[0]
plt.figure(figsize=(5,3)); plt.bar(range(3),mmr); plt.title('MMR penalizes redundancy')
assert int(np.argmax(mmr[1:]))+1==2

In [ ]:
source_len=120; summary_lens=np.array([20,40,80]); ratios=summary_lens/source_len
plt.figure(figsize=(5,3)); plt.bar(['20','40','80'],ratios); plt.title('Compression ratio controls brevity')
assert abs(ratios[0]-1/6)<1e-9

In [ ]:
ref=set('a b c d e'.split()); cand=set('a b c'.split()); rouge=len(ref&cand)/len(ref)
plt.figure(figsize=(4,3)); plt.bar(['ROUGE recall'],[rouge]); plt.ylim(0,1); plt.title('Summary must cover reference facts')
assert abs(rouge-0.6)<1e-9

In [ ]:
facts=np.array([1,1,0,1]); generated=np.array([1,0,0,1]); coverage=(facts&generated).sum()/facts.sum()
plt.figure(figsize=(5,2)); plt.imshow([facts,generated],aspect='auto',cmap='Greens'); plt.yticks([0,1],['source facts','summary']); plt.title('Abstractive fusion should preserve facts')
assert abs(coverage-2/3)<1e-9